# Phase 0 — Lesson 07: Docker for AI
> Containers make "works on my machine" a thing of the past.

**Source:** ai-engineering-from-scratch -> Phase 0 (Setup & Tooling), Lesson 07
**Type:** Build | **Languages:** Docker | **Time:** ~60 minutes
**Prerequisites:** Phase 0, Lessons 01 and 03

This lesson explains *why* Docker solves AI's dependency nightmare, *how* images and layers work, and *when* to use volumes, GPU passthrough, and multi-service orchestration. Run the code cells to see visualizations of the concepts.


# The Problem: "Works on My Machine"

---


You trained a model on your laptop with PyTorch 2.3, CUDA 12.4, and Python 3.12.
Your colleague has PyTorch 2.1, CUDA 11.8, and Python 3.10.
Your model crashes on their machine.

AI projects are dependency nightmares. A typical stack includes:
- Python and its version
- PyTorch, JAX, or TensorFlow
- CUDA drivers and cuDNN
- System-level C libraries
- Specialized packages like flash-attn that need exact compiler versions

Docker packages **all of this** into a single image that runs identically everywhere.


# Visual: Without Docker vs With Docker

---


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Without Docker
ax = axes[0]
ax.axis('off')
ax.text(0.5, 0.95, 'WITHOUT Docker', ha='center', fontsize=13, fontweight='bold', color='#e74c3c')

machines = [
    (0.15, 'Your Machine\nPython 3.12\nCUDA 12.4\nPyTorch 2.3'),
    (0.5, 'Their Machine\nPython 3.10\nCUDA 11.8\nPyTorch 2.1'),
    (0.85, 'Server\nPython 3.11\nCUDA 12.1\nPyTorch 2.2'),
]
for x, label in machines:
    rect = patches.Rectangle((x-0.12, 0.3), 0.24, 0.4, facecolor='#e74c3c', edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(x, 0.5, label, ha='center', va='center', color='white', fontsize=8, fontweight='bold')

for x in [0.15, 0.5, 0.85]:
    ax.annotate('', xy=(x, 0.22), xytext=(x, 0.32),
                arrowprops=dict(arrowstyle='->', color='black', lw=2))
    ax.text(x, 0.12, 'CRASHES', ha='center', fontsize=10, fontweight='bold', color='#e74c3c')

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

# With Docker
ax = axes[1]
ax.axis('off')
ax.text(0.5, 0.95, 'WITH Docker — Same Image Everywhere', ha='center', fontsize=13, fontweight='bold', color='#27ae60')

for i, x in enumerate([0.15, 0.5, 0.85]):
    rect = patches.Rectangle((x-0.12, 0.3), 0.24, 0.4, facecolor='#27ae60', edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(x, 0.5, f'Machine {i+1}\nPython 3.12\nCUDA 12.4\nPyTorch 2.3\nYour Code',
            ha='center', va='center', color='white', fontsize=8, fontweight='bold')

ax.text(0.5, 0.12, 'IDENTICAL ENVIRONMENT', ha='center', fontsize=10, fontweight='bold', color='#27ae60')

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()


> **Without Docker:** Each machine has different versions. Code crashes everywhere.
> **With Docker:** One image runs identically on every machine.


# The Concept: What Docker Actually Is

---


Docker wraps your code, runtime, libraries, and system tools into an isolated unit called a **container**.

Think of it as a lightweight virtual machine — but instead of running its own full operating system, it shares the host OS kernel. That's why containers start in **seconds** instead of **minutes**.

| Virtual Machine | Docker Container |
|-----------------|------------------|
| Runs its own OS kernel | Shares host OS kernel |
| Minutes to start | Seconds to start |
| Gigabytes per VM | Megabytes per container |
| Heavy isolation | Lightweight isolation |


# Key Vocabulary: Image, Container, Dockerfile, Volume

---


| Term | Analogy | What It Actually Means |
|------|---------|----------------------|
| **Image** | A recipe | A read-only template. Built from a Dockerfile. |
| **Container** | A meal cooked from the recipe | A running instance of an image. |
| **Dockerfile** | The recipe itself | Step-by-step instructions to build an image, layer by layer. |
| **Volume** | A pantry shared across meals | Persistent storage that survives container restarts. |
| **docker-compose** | A dinner party plan | A tool for defining multi-container applications in YAML. |


# Why AI Projects Need Docker More Than Most

---


1. **GPU drivers are fragile.** CUDA 12.4 code does not run on CUDA 11.8. Docker isolates the CUDA toolkit inside the container while sharing the host GPU driver.

2. **Model weights are large.** A 7B parameter model is 14 GB in fp16. You do not want to re-download it every time you rebuild. Docker volumes mount your models directory.

3. **Multi-service architectures are common.** A real AI application is not just a Python script. It is an inference server, a vector database for RAG, maybe a web frontend. Docker Compose orchestrates all of these with one command.


# Common Container Patterns in AI

---


| Pattern | What's Inside | Used For |
|---------|---------------|----------|
| Dev Container | Full toolkit, editor support, Jupyter, debugging tools | Development and experimentation |
| Training Container | Minimal — just training script and dependencies | GPU clusters, no editor |
| Inference Container | Optimized for serving, small image, fast cold start | Production behind a load balancer |


# Visual: Three AI Container Patterns

---


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))

patterns = [
    ('Dev Container', 'Full toolkit\nJupyter, debugger, editor\nUsed during development', '#3498db'),
    ('Training Container', 'Minimal\nTraining script + deps\nRuns on GPU clusters', '#2ecc71'),
    ('Inference Container', 'Serving-optimized\nSmall image, fast start\nProduction behind load balancer', '#e74c3c'),
]

for i, (title, desc, color) in enumerate(patterns):
    ax = axes[i]
    ax.axis('off')
    rect = plt.Rectangle((0.1, 0.2), 0.8, 0.6, facecolor=color, edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(0.5, 0.8, title, ha='center', va='center', color='white', fontweight='bold', fontsize=12)
    ax.text(0.5, 0.45, desc, ha='center', va='center', fontsize=9, color='white', style='italic')

plt.tight_layout()
plt.show()


> **Choose the pattern for the job.** Dev for experimenting, training for clusters, inference for production.


# Choosing the Right Base Image

---


Your `FROM` line determines what is pre-installed. Choose wisely — it affects build time and image size.

| Base Image | What It Includes | Best For | Size |
|------------|-----------------|----------|------|
| `nvidia/cuda:12.4.1-devel-ubuntu22.04` | Full CUDA toolkit + compilers | Building packages that need nvcc (flash-attn, bitsandbytes) | ~4 GB |
| `nvidia/cuda:12.4.1-runtime-ubuntu22.04` | CUDA runtime only | Running pre-built code | ~1.5 GB |
| `pytorch/pytorch:2.6.0-cuda12.4-cudnn9-runtime` | PyTorch pre-installed on CUDA | Skipping PyTorch install step | ~6 GB |
| `python:3.12-slim` | No CUDA, CPU only | CPU inference, lightweight tools | ~150 MB |


# Visual: Base Image Size Comparison

---


In [ ]:
import matplotlib.pyplot as plt

images = ['python:3.12-slim\n(CPU only)', 'nvidia/cuda:12.4.1-runtime', 'nvidia/cuda:12.4.1-devel', 'pytorch/pytorch:2.6.0']
sizes_mb = [150, 1500, 4000, 6000]
colors = ['#2ecc71', '#f39c12', '#e67e22', '#e74c3c']

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(images, sizes_mb, color=colors, alpha=0.8)
ax.set_ylabel('Size (MB)', fontsize=10)
ax.set_title('Docker Base Image Sizes', fontsize=12, fontweight='bold')
ax.set_yscale('log')
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=15, ha='right')

for bar, mb in zip(bars, sizes_mb):
    ax.text(bar.get_x() + bar.get_width()/2, mb * 1.1, f'{mb:,} MB', ha='center', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.show()


> **Smaller is faster to build and push.** Use `python:3.12-slim` for CPU-only tools. Use `nvidia/cuda:...-devel` only when you need to compile CUDA extensions.


# Step 1: Install Docker

---


**macOS:**
```bash
brew install --cask docker
open /Applications/Docker.app
```

**Ubuntu:**
```bash
curl -fsSL https://get.docker.com | sh
sudo usermod -aG docker $USER
# Log out and back in for group change to take effect
```

**Verify:**
```bash
docker --version
docker run hello-world
```


# Step 2: NVIDIA Container Toolkit (Linux with NVIDIA GPU)

---


This lets Docker containers access your GPU. macOS and Windows (WSL2) users can skip this; Docker Desktop handles GPU passthrough differently on those platforms.

**The key insight:** The CUDA toolkit lives *inside* the container, but the GPU **driver** is shared from the host.

```bash
distribution=$(. /etc/os-release;echo $ID$VERSION_ID)
curl -fsSL https://nvidia.github.io/libnvidia-container/gpgkey | sudo gpg --dearmor -o /usr/share/keyrings/nvidia-container-toolkit-keyring.gpg
curl -s -L https://nvidia.github.io/libnvidia-container/$distribution/libnvidia-container.list | \
    sed 's#deb https://#deb [signed-by=/usr/share/keyrings/nvidia-container-toolkit-keyring.gpg] https://#g' | \
    sudo tee /etc/apt/sources.list.d/nvidia-container-toolkit.list

sudo apt-get update
sudo apt-get install -y nvidia-container-toolkit
sudo nvidia-ctk runtime configure --runtime=docker
sudo systemctl restart docker
```

**Test GPU access inside a container:**
```bash
docker run --rm --gpus all nvidia/cuda:12.4.1-base-ubuntu22.04 nvidia-smi
```


# Visual: How GPU Passthrough Works

---


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(10, 4))
ax.axis('off')

# Host
rect_host = patches.Rectangle((0.05, 0.3), 0.9, 0.6, facecolor='#2c3e50', edgecolor='black', linewidth=2)
ax.add_patch(rect_host)
ax.text(0.5, 0.7, 'HOST MACHINE', ha='center', va='center', color='white', fontweight='bold', fontsize=12)
ax.text(0.15, 0.55, 'NVIDIA Driver', ha='center', fontsize=9, color='white')
ax.text(0.5, 0.55, 'GPU Hardware', ha='center', fontsize=9, color='white')
ax.text(0.85, 0.55, 'Container Toolkit', ha='center', fontsize=9, color='white')

# Container
rect_cont = patches.Rectangle((0.2, 0.05), 0.6, 0.2, facecolor='#3498db', edgecolor='black', linewidth=2)
ax.add_patch(rect_cont)
ax.text(0.5, 0.15, 'CONTAINER', ha='center', va='center', color='white', fontweight='bold', fontsize=11)
ax.text(0.25, 0.08, 'CUDA Toolkit (inside)', fontsize=8, color='white')
ax.text(0.5, 0.08, 'PyTorch', fontsize=8, color='white')
ax.text(0.75, 0.08, 'your code', fontsize=8, color='white')

# Arrow showing passthrough
ax.annotate('', xy=(0.5, 0.28), xytext=(0.5, 0.25),
            arrowprops=dict(arrowstyle='->', color='#27ae60', lw=3))
ax.text(0.5, 0.31, '--gpus all shares host GPU driver', ha='center', fontsize=9, color='#27ae60', fontweight='bold')

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title("GPU Passthrough: Host Driver Shared with Container", fontsize=12, fontweight="bold", pad=20)
plt.tight_layout()
plt.show()


> **macOS / Apple Silicon:** No CUDA on Mac. Run lessons natively with MPS (from Lesson 01). Docker images are for x86_64 Linux hosts with NVIDIA GPUs.


# Step 4: Write a Dockerfile for AI Development

---


Here is the Dockerfile. Each instruction creates a **layer** — unchanged layers are cached, so rebuilds are fast.

```dockerfile
FROM --platform=linux/amd64 nvidia/cuda:12.4.1-devel-ubuntu22.04

ENV DEBIAN_FRONTEND=noninteractive
ENV PYTHONUNBUFFERED=1

RUN apt-get update && apt-get install -y --no-install-recommends \
    software-properties-common git curl build-essential \
    && add-apt-repository -y ppa:deadsnakes/ppa \
    && apt-get update && apt-get install -y --no-install-recommends \
    python3.12 python3.12-venv python3.12-dev \
    && rm -rf /var/lib/apt/lists/*

RUN update-alternatives --install /usr/bin/python python /usr/bin/python3.12 1

RUN python -m pip install --no-cache-dir --upgrade pip setuptools wheel

RUN python -m pip install --no-cache-dir \
    torch==2.6.0+cu124 torchvision==0.21.0+cu124 torchaudio==2.6.0+cu124 \
    --index-url https://download.pytorch.org/whl/cu124

RUN python -m pip install --no-cache-dir \
    numpy pandas scikit-learn matplotlib jupyter \
    transformers datasets accelerate safetensors

WORKDIR /workspace
VOLUME ["/workspace", "/models"]
EXPOSE 8888
CMD ["python"]
```

**Build it:**
```bash
docker build -t ai-dev -f phases/00-setup-and-tooling/07-docker-for-ai/code/Dockerfile .
```

First build takes a while (downloading CUDA base image + PyTorch). Subsequent builds use cached layers.


# Visual: Docker Image Layers and Caching

---


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(10, 4))
ax.axis('off')

layers = [
    ('FROM cuda:12.4.1-devel', '#2c3e50', 'Base image'),
    ('apt-get install python3.12', '#34495e', 'System packages'),
    ('pip install torch+cu124', '#3498db', 'PyTorch (large)'),
    ('pip install numpy, jupyter...', '#2ecc71', 'AI libraries'),
    ('WORKDIR /workspace', '#f39c12', 'Working dir'),
    ('VOLUME, EXPOSE, CMD', '#e67e22', 'Metadata'),
]

y = 0.85
for label, color, desc in layers:
    rect = patches.Rectangle((0.1, y-0.08), 0.8, 0.14, facecolor=color, edgecolor='black', linewidth=1)
    ax.add_patch(rect)
    ax.text(0.15, y, label, ha='left', va='center', color='white', fontweight='bold', fontsize=10)
    ax.text(0.88, y, desc, ha='right', va='center', fontsize=8, style='italic')
    y -= 0.18

ax.annotate('', xy=(0.45, 0.02), xytext=(0.45, 0.14),
            arrowprops=dict(arrowstyle='<->', color='#e74c3c', lw=2))
ax.text(0.5, 0.06, 'Change line 4 → rebuild from here down\nLines 1-3 use cache', ha='center', fontsize=9, color='#e74c3c')

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title("Docker Layers: Each Instruction Is a Cached Layer", fontsize=12, fontweight="bold", pad=20)
plt.tight_layout()
plt.show()


> **Layer caching:** Change a line near the bottom → only that layer and below rebuilds. Change `FROM` → everything rebuilds. Put slow-changing steps (base image, system packages) at the top.


# Run the Container

---


**Basic run with GPU and volume mounts:**
```bash
docker run --rm -it --gpus all \
    -v $(pwd):/workspace \
    -v ~/models:/models \
    ai-dev python -c "import torch; print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')"
```

**Run Jupyter inside the container:**
```bash
docker run --rm -it --gpus all \
    -v $(pwd):/workspace \
    -v ~/models:/models \
    -p 8888:8888 \
    ai-dev jupyter notebook --ip=0.0.0.0 --port=8888 --no-browser --allow-root
```

`-v $(pwd):/workspace` mounts your code so changes on the host appear inside the container.


# Step 5: Volume Mounts for Data and Models

---


Volume mounts are critical for AI work. Without them, your 14 GB model downloads vanish when the container stops.

```bash
# Mount your code
-v $(pwd):/workspace

# Mount a shared models directory
-v ~/models:/models

# Mount datasets
-v ~/datasets:/data
```

Inside your training script, load from the mounted path:
```python
from transformers import AutoModel

model = AutoModel.from_pretrained("/models/llama-7b")
```

The model lives on your host filesystem. Rebuild the container as often as you want without re-downloading.


# Visual: Volume Persistence Across Rebuilds

---


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(10, 4))
ax.axis('off')

# Container 1
rect_c1 = patches.Rectangle((0.1, 0.4), 0.3, 0.4, facecolor='#e74c3c', edgecolor='black', linewidth=2)
ax.add_patch(rect_c1)
ax.text(0.25, 0.7, 'Container 1', ha='center', va='center', color='white', fontweight='bold')
ax.text(0.25, 0.5, 'stops\ndeleted', ha='center', va='center', fontsize=8)

# Volume (host)
rect_v = patches.Rectangle((0.45, 0.25), 0.2, 0.6, facecolor='#27ae60', edgecolor='black', linewidth=2)
ax.add_patch(rect_v)
ax.text(0.55, 0.55, 'HOST VOLUME', ha='center', va='center', color='white', fontweight='bold')
ax.text(0.55, 0.35, '/models\nllama-7b\n(14 GB)', ha='center', va='center', fontsize=8)

# Container 2 (rebuilt)
rect_c2 = patches.Rectangle((0.7, 0.4), 0.3, 0.4, facecolor='#3498db', edgecolor='black', linewidth=2)
ax.add_patch(rect_c2)
ax.text(0.85, 0.7, 'Container 2', ha='center', va='center', color='white', fontweight='bold')
ax.text(0.85, 0.5, 'rebuilt\nmodel still there', ha='center', va='center', fontsize=8)

# Arrows
ax.annotate('', xy=(0.4, 0.45), xytext=(0.26, 0.45),
            arrowprops=dict(arrowstyle='->', color='#27ae60', lw=2))
ax.annotate('', xy=(0.66, 0.45), xytext=(0.65, 0.45),
            arrowprops=dict(arrowstyle='->', color='#27ae60', lw=2))

ax.text(0.5, 0.08, 'Volume survives container rebuilds — model is not re-downloaded', ha='center',
        fontsize=10, fontweight='bold', color='#27ae60')

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title("Volume Persistence: Models Survive Container Rebuilds", fontsize=12, fontweight="bold", pad=20)
plt.tight_layout()
plt.show()


> **Without volumes:** Every container rebuild loses everything inside. 14 GB model re-downloaded each time.
> **With volumes:** Model lives on host. Container rebuilds are instant for data.


# Step 6: Docker Compose for Multi-Service AI Apps

---


A real RAG application needs an inference server **and** a vector database. Docker Compose runs both with one command.

See `code/docker-compose.yml`:
```yaml
services:
  ai-dev:
    build:
      context: .
      dockerfile: Dockerfile
    deploy:
      resources:
        reservations:
          devices:
            - driver: nvidia
              count: all
              capabilities: [gpu]
    volumes:
      - ../../../:/workspace
      - ~/models:/models
      - ~/datasets:/data
    ports:
      - "8888:8888"
    stdin_open: true
    tty: true
    command: jupyter notebook --ip=0.0.0.0 --port=8888 --no-browser --allow-root

  qdrant:
    image: qdrant/qdrant:v1.12.5
    ports:
      - "6333:6333"
      - "6334:6334"
    volumes:
      - qdrant_data:/qdrant/storage

volumes:
  qdrant_data:
```

**Start everything:**
```bash
cd phases/00-setup-and-tooling/07-docker-for-ai/code
docker compose up -d
```

Docker Compose creates a shared network automatically. Your AI container reaches Qdrant at `http://qdrant:6333` by service name.


# Visual: Docker Compose Multi-Service Network

---


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(9, 4))
ax.axis('off')

# AI container
rect_ai = patches.Rectangle((0.1, 0.3), 0.35, 0.5, facecolor='#3498db', edgecolor='black', linewidth=2)
ax.add_patch(rect_ai)
ax.text(0.275, 0.7, 'ai-dev', ha='center', va='center', color='white', fontweight='bold', fontsize=11)
ax.text(0.275, 0.5, 'Jupyter + PyTorch', ha='center', va='center', fontsize=8, color='white')
ax.text(0.275, 0.38, 'port 8888', ha='center', va='center', fontsize=8, color='white')

# Qdrant container
rect_q = patches.Rectangle((0.55, 0.3), 0.35, 0.5, facecolor='#2ecc71', edgecolor='black', linewidth=2)
ax.add_patch(rect_q)
ax.text(0.725, 0.7, 'qdrant', ha='center', va='center', color='white', fontweight='bold', fontsize=11)
ax.text(0.725, 0.5, 'Vector Database', ha='center', va='center', fontsize=8, color='white')
ax.text(0.725, 0.38, 'port 6333', ha='center', va='center', fontsize=8, color='white')

# Network between them
ax.annotate('', xy=(0.46, 0.55), xytext=(0.44, 0.55),
            arrowprops=dict(arrowstyle='<->', color='#e74c3c', lw=3))
ax.text(0.5, 0.6, 'shared network', ha='center', fontsize=9, color='#e74c3c', fontweight='bold')
ax.text(0.5, 0.52, 'ai-dev reaches qdrant at http://qdrant:6333', ha='center', fontsize=9, style='italic')

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title("Docker Compose: Services Talk by Name on Shared Network", fontsize=12, fontweight="bold", pad=20)
plt.tight_layout()
plt.show()


> **Service name as hostname:** `ai-dev` connects to `qdrant` using `http://qdrant:6333`. No IP addresses needed.


# Step 7: Useful Docker Commands for AI Work

---


```bash
# List running containers
docker ps

# List all images and their sizes
docker images

# Remove unused images (reclaim disk space)
docker system prune -a

# Check GPU usage inside a running container
docker exec -it <container_id> nvidia-smi

# Copy a file from container to host
docker cp <container_id>:/workspace/results.csv ./results.csv

# View container logs
docker logs -f <container_id>

# Stop and remove compose services
docker compose down
docker compose down -v   # also deletes volumes
```


# Quiz Review

---


Try answering these before reading the explanations:

**Q1:** What is the primary difference between a Docker container and a virtual machine?
> **Explanation:** Containers share the host OS kernel while VMs run their own full OS. Containers start in seconds; VMs take minutes.

**Q2:** What is a Dockerfile?
> **Explanation:** A set of instructions for building a Docker image layer by layer. Each instruction (FROM, RUN, COPY, etc.) creates a cached layer.

**Q3:** Why are volume mounts critical for AI development with Docker?
> **Explanation:** Volumes persist data (models, datasets, code) across container rebuilds. Without them, everything inside a container is lost when it stops.

**Q4:** What does the NVIDIA Container Toolkit enable?
> **Explanation:** Exposing host GPUs to Docker containers via the `--gpus` flag. The CUDA toolkit lives inside the container; the GPU driver is shared from the host.

**Q5:** In a Docker Compose file for AI, how does the 'ai-dev' service reach the 'qdrant' vector database?
> **Explanation:** By using the service name 'qdrant' as the hostname, since Compose creates a shared network.


# Practice Exercises

---


Do these in order:

**Exercise 1:** Build the Dockerfile and run `python -c "import torch; print(torch.__version__)"` inside the container.

**Exercise 2:** Start the docker-compose stack and verify Qdrant is accessible from the AI container at `http://qdrant:6333/collections`.

**Exercise 3:** Add `flask` to the Dockerfile, rebuild, and run a simple API server on port 5000. Map the port with `-p 5000:5000`.

**Exercise 4:** Measure the image size with `docker images`. Try switching the base image from `devel` to `runtime` and compare sizes.

### No GPU?
Remove the `--gpus all` flag and the NVIDIA deploy block. The container still works for CPU-based lessons. PyTorch detects the absence of CUDA and falls back to CPU automatically.


# Key Terms

---


| Term | What people say | What it actually means |
|------|----------------|----------------------|
| Container | "Lightweight VM" | An isolated process using the host kernel, with its own filesystem and network |
| Image layer | "Cached step" | Each Dockerfile instruction creates a layer. Unchanged layers are cached, so rebuilds are fast. |
| NVIDIA Container Toolkit | "GPU in Docker" | A runtime hook that exposes host GPUs to containers via `--gpus` flag |
| Volume mount | "Shared folder" | A directory on the host mapped into the container. Changes persist after the container stops. |
| Base image | "Starting point" | The `FROM` image your Dockerfile builds on top of. Determines what is pre-installed. |


# Next Steps

---


When your Docker environment is running, continue to:
**Lesson 08: Editor Setup**

Your `ai-engineering-from-scratch` learning directory is **completely untouched** — these notes live separately in this folder.
